# 2.1 Bloom Experiments – B1 Tuning + B2 Embedding

**Tác giả:** Đăng (TV1)  
**Mục tiêu:**
- GridSearchCV C ∈ {0.1, 0.5, 1.0, 5.0, 10.0} cho B1
- Thêm n-gram variants (char 3-5)
- B2: multilingual-e5-small embedding (fallback TF-IDF SVD)
- Vẽ learning curve
- Bảng so sánh B0→B2

In [ ]:
import json
import pickle
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedGroupKFold, GridSearchCV, learning_curve, cross_val_score
)
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

ROOT = Path('..').resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
with open(ROOT / 'config' / 'params.yaml') as f:
    CFG = yaml.safe_load(f)

RANDOM_SEED = CFG.get('random_seed', 42)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = ROOT / 'models'
METRICS_PATH = ROOT / 'results' / 'metrics.json'

BLOOM_NAMES = ['Ngoài miền','Nhớ','Hiểu','Áp dụng','Phân tích','Đánh giá','Sáng tạo']
print('OK')

In [ ]:
# Tái sử dụng dữ liệu từ notebook 2.0
CLO_CSV = ROOT / 'data' / 'processed' / 'clos_labeled.csv'
if CLO_CSV.exists():
    df = pd.read_csv(CLO_CSV)
else:
    # Dùng synthetic 28 CLO (giống notebook 2.0)
    SYNTHETIC_CLOS = [
        ('841020', 'Trình bày định nghĩa biến và kiểu dữ liệu', 1),
        ('841020', 'Liệt kê các toán tử cơ bản trong Python', 1),
        ('841020', 'Giải thích cách hoạt động của vòng lặp for', 2),
        ('841020', 'Mô tả sự khác biệt giữa list và tuple', 2),
        ('841020', 'Viết chương trình tính giai thừa bằng đệ quy', 3),
        ('841020', 'Áp dụng hàm lambda để xử lý danh sách', 3),
        ('841303', 'Nêu định nghĩa cấu trúc stack và queue', 1),
        ('841303', 'Giải thích độ phức tạp thuật toán O(n log n)', 2),
        ('841303', 'Cài đặt cây nhị phân tìm kiếm bằng Java', 3),
        ('841303', 'Phân tích ưu nhược điểm của các thuật toán sắp xếp', 4),
        ('841303', 'Lựa chọn cấu trúc dữ liệu phù hợp cho bài toán cụ thể', 5),
        ('841109', 'Kể tên các thành phần cơ bản của mạng máy tính', 1),
        ('841109', 'Giải thích mô hình OSI 7 tầng', 2),
        ('841109', 'Cấu hình router và switch cơ bản', 3),
        ('841109', 'So sánh TCP và UDP về độ tin cậy và hiệu năng', 4),
        ('841109', 'Thiết kế mạng LAN cho văn phòng 50 người', 6),
        ('841434', 'Định nghĩa khái niệm transaction trong CSDL', 1),
        ('841434', 'Mô tả mô hình quan hệ và các ràng buộc toàn vẹn', 2),
        ('841434', 'Viết câu truy vấn SQL JOIN phức tạp', 3),
        ('841434', 'Phân tích và chuẩn hoá lược đồ CSDL đến 3NF', 4),
        ('841434', 'Đánh giá hiệu năng truy vấn và đề xuất tối ưu index', 5),
        ('841470', 'Trình bày nguyên lý hoạt động của mạng nơ-ron', 1),
        ('841470', 'Giải thích thuật toán gradient descent', 2),
        ('841470', 'Huấn luyện mô hình phân loại đơn giản bằng scikit-learn', 3),
        ('841470', 'Phân tích kết quả confusion matrix và đề xuất cải tiến', 4),
        ('841470', 'Đánh giá và lựa chọn mô hình ML phù hợp cho bài toán thực', 5),
        ('841470', 'Thiết kế pipeline ML end-to-end cho bài toán phân loại văn bản', 6),
        ('841020', 'Có ý thức bảo vệ quyền sở hữu trí tuệ phần mềm', 0),
    ]
    df = pd.DataFrame(SYNTHETIC_CLOS, columns=['course_code', 'text', 'bloom_level'])

df['bloom_level'] = df['bloom_level'].astype(int)
if 'course_code' not in df.columns:
    df['course_code'] = 'unknown'

X = df['text'].values
y = df['bloom_level'].values
groups = df['course_code'].values
print(f'Dữ liệu: {len(df)} CLO, {len(np.unique(groups))} môn')

## 1. B1 – GridSearchCV tuning C

In [ ]:
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, analyzer='word')
tfidf_char = TfidfVectorizer(ngram_range=(2, 4), max_features=3000, analyzer='char_wb')
tfidf_word.fit(X); tfidf_char.fit(X)
X_feat = sp.hstack([tfidf_word.transform(X), tfidf_char.transform(X)])

cv = StratifiedGroupKFold(n_splits=min(5, len(np.unique(groups))), shuffle=True, random_state=RANDOM_SEED)

# GridSearch C
C_values = [0.1, 0.5, 1.0, 5.0, 10.0]
results_grid = {}
for C in C_values:
    clf = LogisticRegression(C=C, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)
    scores = cross_val_score(clf, X_feat, y, groups=groups, cv=cv, scoring='f1_macro')
    results_grid[C] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'C={C:5.1f}: Macro-F1 = {scores.mean():.4f} ± {scores.std():.4f}')

best_C = max(results_grid, key=lambda c: results_grid[c]['mean'])
print(f'\nBest C = {best_C} (Macro-F1 = {results_grid[best_C]["mean"]:.4f})')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
means = [results_grid[C]['mean'] for C in C_values]
stds = [results_grid[C]['std'] for C in C_values]
ax.errorbar([str(C) for C in C_values], means, yerr=stds, marker='o',
            linewidth=2, capsize=5, color='#1565C0')
ax.axhline(0.75, color='red', linestyle='--', linewidth=1.2, label='Target (0.75)')
ax.set_xlabel('C (Regularization)', fontsize=11)
ax.set_ylabel('Macro-F1 (CV)', fontsize=11)
ax.set_title('B1: GridSearchCV – Tuning C', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(alpha=0.4, linestyle='--')
plt.tight_layout()
plt.savefig(FIG_DIR / 'm2_b1_gridsearch.png', dpi=150, bbox_inches='tight')
plt.close()
print('Đã lưu: reports/figures/m2_b1_gridsearch.png')

## 2. B2 – Multilingual E5 Embedding + LogReg

In [ ]:
# Thử dùng multilingual-e5-small; fallback TF-IDF SVD nếu không load được
try:
    from sentence_transformers import SentenceTransformer
    EMBED_MODEL = CFG.get('m3_embedding', {}).get('model', 'intfloat/multilingual-e5-small')
    print(f'Tải model: {EMBED_MODEL} ...')
    encoder = SentenceTransformer(EMBED_MODEL)
    # Thêm prefix 'query: ' theo yêu cầu e5
    X_emb = encoder.encode(['query: ' + t for t in X], normalize_embeddings=True, show_progress_bar=True)
    b2_method = 'multilingual-e5-small'
    print(f'Embedding shape: {X_emb.shape}')
except Exception as e:
    print(f'Không load được model embedding: {e}')
    print('Fallback: TF-IDF SVD (50 chiều)')
    tfidf_full = TfidfVectorizer(ngram_range=(1, 3), max_features=10000, analyzer='word')
    svd = TruncatedSVD(n_components=50, random_state=RANDOM_SEED)
    X_tfidf = tfidf_full.fit_transform(X)
    X_emb = svd.fit_transform(X_tfidf)
    b2_method = 'tfidf-svd-50d'

# Kết hợp embedding + tfidf features
X_feat_emb = sp.hstack([sp.csr_matrix(X_emb), X_feat])

# 5-fold CV với best_C
clf_b2 = LogisticRegression(C=best_C, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)
scores_b2 = cross_val_score(clf_b2, X_feat_emb, y, groups=groups, cv=cv, scoring='f1_macro')
b2_cv_mean = scores_b2.mean()
b2_cv_std = scores_b2.std()
print(f'B2 ({b2_method}) 5-fold CV Macro-F1: {b2_cv_mean:.4f} ± {b2_cv_std:.4f}')

## 3. Learning Curve

In [ ]:
from src.syllabus_core.visualization.visualize import plot_learning_curve

clf_lc = LogisticRegression(C=best_C, class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)

# Với dữ liệu nhỏ, train_sizes linh hoạt
n_samples = len(X)
train_sizes_pct = np.linspace(0.2, 1.0, min(5, n_samples // 2))

try:
    from sklearn.model_selection import learning_curve as lc_func
    sizes, train_sc, val_sc = lc_func(
        clf_lc, X_feat, y, groups=groups,
        train_sizes=train_sizes_pct,
        cv=cv, scoring='f1_macro',
        n_jobs=1,
    )
    plot_learning_curve(
        sizes, train_sc, val_sc,
        title='Learning Curve – B1 TF-IDF + LogReg',
        save_path=str(FIG_DIR / 'm2_learning_curve.png'),
        metric_name='Macro-F1',
    )
    print('Đã lưu: reports/figures/m2_learning_curve.png')
except Exception as e:
    print(f'Lỗi learning curve: {e}')

## 4. Bảng so sánh B0 → B2

In [ ]:
# Đọc B0 từ metrics
with open(METRICS_PATH, encoding='utf-8') as f:
    metrics = json.load(f)

b0_f1 = metrics['m2_bloom']['experiments']['b0_rule_baseline'].get('cv_macro_f1_mean', 0.0)
b1_f1 = metrics['m2_bloom']['experiments']['b1_tfidf_logreg'].get('cv_macro_f1_mean', 0.0)

summary = pd.DataFrame([
    {'Model': 'B0 Rule-based', 'Method': 'Tra từ điển động từ', 'Macro-F1 CV': b0_f1, 'Ghi chú': 'Không cần train'},
    {'Model': f'B1 Best (C={best_C})', 'Method': 'TF-IDF + LogReg', 'Macro-F1 CV': b1_f1, 'Ghi chú': '5-fold CV'},
    {'Model': f'B2 ({b2_method})', 'Method': 'Embedding + LogReg', 'Macro-F1 CV': b2_cv_mean, 'Ghi chú': '5-fold CV'},
])

print('=== Bảng so sánh B0 → B2 ===')
print(summary.to_string(index=False))

# Cập nhật metrics
metrics['m2_bloom']['experiments']['b2_e5_embedding']['cv_macro_f1_mean'] = round(float(b2_cv_mean), 4)
metrics['m2_bloom']['experiments']['b2_e5_embedding']['cv_macro_f1_std'] = round(float(b2_cv_std), 4)
metrics['m2_bloom']['experiments']['b2_e5_embedding']['method'] = b2_method
metrics['m2_bloom']['experiments']['b2_e5_embedding']['status'] = 'completed'

# Chọn best model
best_f1 = max(b0_f1, b1_f1, b2_cv_mean)
metrics['m2_bloom']['best_macro_f1'] = round(float(best_f1), 4)
metrics['m2_bloom']['target_exceeded'] = best_f1 >= metrics['m2_bloom']['target_macro_f1']

with open(METRICS_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f'\nBest Macro-F1 = {best_f1:.4f}')
target = metrics['m2_bloom']['target_macro_f1']
print(f'Target {target}: {"✅ ĐẠT" if best_f1 >= target else "❌ CHƯA ĐẠT"}')